In [1]:
import sys

sys.path.insert(0, "../")

In [2]:
from mdu.unc.risk_metrics import RiskType, GName, ApproximationType
from mdu.unc.constants import UncertaintyType
from mdu.data.constants import DatasetName

from mdu.eval.eval_utils import get_results_path, compute_ood_detection_metrics, compute_misclassification_detection_metrics, compute_selective_prediction_metrics

from mdu.eval.eval_utils import (
    create_output_filename,
    load_predictions_and_split,
)
import numpy as np

import pandas as pd
from tqdm.auto import tqdm

/home/nkotelevskii/github/multidimensional_uncertainty/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_uncertainty_additive_measure(
    ind_dataset,
    ood_dataset,
    metrics_list,
    prediction_data,
    results,
    pbar,
    processed_same_dataset,
):
    """Process a single uncertainty measure configuration"""

    try:
        # Load uncertainty measure data
        uncertainty_data = None
        for (
            uncertainty_type,
            gname,
            risk_type,
            gt_approx,
            pred_approx) in metrics_list:

            path = get_results_path(
                ind_dataset,
                ood_dataset,
                uncertainty_type,
                gname,
                risk_type,
                gt_approx,
                pred_approx,
                "../resources/results_cleaned",
            )
            aux_data_ = np.load(path)
            if uncertainty_data is None:
                uncertainty_data = {}
                uncertainty_data["ind_calib"] = aux_data_["ind_calib"]
                uncertainty_data["ind_test"] = aux_data_["ind_test"]
                uncertainty_data["ood"] = aux_data_["ood"]
            else:
                uncertainty_data["ind_calib"] = uncertainty_data["ind_calib"] + aux_data_["ind_calib"]
                uncertainty_data["ind_test"] = uncertainty_data["ind_test"] + aux_data_["ind_test"]
                uncertainty_data["ood"] = uncertainty_data["ood"] + aux_data_["ood"]

        # Create measure identifier
        measure_id = "naive_additive"

        # Get prediction data
        pred_data = prediction_data.get(ind_dataset)

        if pred_data is None:
            pbar.update(1)
            return

        # Process each ensemble group
        for group_key, group_data in pred_data.items():
            group_idx = int(group_key.split("_")[1])  # Extract group index

            # Extract uncertainty scores for this group
            ind_test_scores = uncertainty_data["ind_test"][
                group_idx, 0, :
            ]  # Shape: (n_test_samples,)
            ind_calib_scores = uncertainty_data["ind_calib"][
                group_idx, 0, :
            ]  # Shape: (n_calib_samples,)
            ood_scores = uncertainty_data["ood"][
                group_idx, 0, :
            ]  # Shape: (n_ood_samples,)

            # Get predictions and labels
            y_pred = group_data["y_pred"]
            y_test = group_data["y_test"]

            # Determine problem types to evaluate
            if ind_dataset != ood_dataset:
                # Case 1: OOD Detection (different datasets)
                ood_metrics = compute_ood_detection_metrics(ind_test_scores, ood_scores)

                results.append(
                    {
                        "ind_dataset": ind_dataset.value,
                        "ood_dataset": ood_dataset.value,
                        "measure": measure_id,
                        "gname": gname.value if gname else None,
                        "ensemble_group": group_idx,
                        "problem_type": "ood_detection",
                        "roc_auc": ood_metrics["roc_auc"],
                        "average_precision": None,
                        "accuracy": None,
                        "aurc": None,
                        "acc_cov_auc": None,
                        "coverage_at_1pct_error": None,
                        "coverage_at_2pct_error": None,
                        "coverage_at_5pct_error": None,
                        "n_ind_samples": ood_metrics["n_ind_samples"],
                        "n_ood_samples": ood_metrics["n_ood_samples"],
                        "n_correct": None,
                        "n_incorrect": None,
                        "ensemble_accuracy": group_data["ensemble_accuracy"],
                    }
                )

            # Case 2 & 3: Same dataset evaluation - Use ind_test scores for misclassification and selective prediction
            # Only do this once per (ind_dataset, measure_id, group_idx) combination to avoid duplicates across OOD datasets
            same_dataset_key = (ind_dataset, measure_id, group_idx)
            if same_dataset_key not in processed_same_dataset:
                processed_same_dataset.add(same_dataset_key)

                # Misclassification detection using ind_test scores
                misc_metrics = compute_misclassification_detection_metrics(
                    ind_test_scores, y_pred, y_test
                )

                results.append(
                    {
                        "ind_dataset": ind_dataset.value,
                        "ood_dataset": ind_dataset.value,  # Same dataset for both
                        "measure": measure_id,
                        "ensemble_group": group_idx,
                        "problem_type": "misclassification_detection",
                        "roc_auc": misc_metrics["roc_auc"],
                        "average_precision": misc_metrics["average_precision"],
                        "accuracy": misc_metrics["accuracy"],
                        "aurc": None,
                        "acc_cov_auc": None,
                        "coverage_at_1pct_error": None,
                        "coverage_at_2pct_error": None,
                        "coverage_at_5pct_error": None,
                        "n_ind_samples": len(ind_test_scores),
                        "n_ood_samples": None,
                        "n_correct": misc_metrics["n_correct"],
                        "n_incorrect": misc_metrics["n_incorrect"],
                        "ensemble_accuracy": group_data["ensemble_accuracy"],
                    }
                )

                # Selective prediction using ind_test scores
                sel_metrics = compute_selective_prediction_metrics(
                    ind_test_scores, y_pred, y_test
                )

                results.append(
                    {
                        "ind_dataset": ind_dataset.value,
                        "ood_dataset": ind_dataset.value,  # Same dataset for both
                        "measure": measure_id,
                        "ensemble_group": group_idx,
                        "problem_type": "selective_prediction",
                        "roc_auc": None,
                        "average_precision": None,
                        "accuracy": sel_metrics["overall_accuracy"],
                        "aurc": sel_metrics["aurc"],
                        "acc_cov_auc": sel_metrics["acc_cov_auc"],
                        "coverage_at_1pct_error": sel_metrics["coverage_at_1pct_error"],
                        "coverage_at_2pct_error": sel_metrics["coverage_at_2pct_error"],
                        "coverage_at_5pct_error": sel_metrics["coverage_at_5pct_error"],
                        "n_ind_samples": sel_metrics["n_samples"],
                        "n_ood_samples": None,
                        "n_correct": None,
                        "n_incorrect": None,
                        "ensemble_accuracy": group_data["ensemble_accuracy"],
                    }
                )

    except Exception as e:
        print(e)
        pass

    if pbar is not None:
        pbar.update(1)


In [4]:
datasets_ind = [
    DatasetName.CIFAR10,
    DatasetName.CIFAR100,
    DatasetName.TINY_IMAGENET,
]
datasets_ood_ = [
    DatasetName.CIFAR10,
    DatasetName.CIFAR100,
    DatasetName.TINY_IMAGENET,
    DatasetName.SVHN,
]
datasets_ood_tiny_imagenet_ = [
    DatasetName.TINY_IMAGENET,
    DatasetName.IMAGENET_A,
    DatasetName.IMAGENET_O,
    DatasetName.IMAGENET_R,
]

In [5]:
metrics_list = [
    (UncertaintyType.RISK, GName.LOG_SCORE, RiskType.BAYES_RISK, ApproximationType.OUTER, None),
    (UncertaintyType.RISK, GName.LOG_SCORE, RiskType.EXCESS_RISK, ApproximationType.OUTER, ApproximationType.OUTER),
    (UncertaintyType.RISK, GName.LOG_SCORE, RiskType.TOTAL_RISK, ApproximationType.OUTER, ApproximationType.OUTER),
    (UncertaintyType.MAHALANOBIS, None, None, None, None),
]

In [6]:
def main():
    results = []
    print("Loading predictions for all datasets...")
    prediction_data = {}
    for ind_dataset in datasets_ind:
        try:
            pred_data = load_predictions_and_split(
                ind_dataset, weights_root="../resources/model_weights"
            )
            prediction_data[ind_dataset] = pred_data

        except Exception as e:
            print(f"✗ Failed to load predictions for {ind_dataset.value}: {e}")
            prediction_data[ind_dataset] = None

    # Create progress bar - count regular measures + compositions
    total_combinations = 0
    for ind_dataset in datasets_ind:
        if ind_dataset == DatasetName.TINY_IMAGENET:
            datasets_ood = datasets_ood_tiny_imagenet_
        else:
            datasets_ood = datasets_ood_

        for ood_dataset in datasets_ood:
            total_combinations += 1
            # Add multidimensional compositions

    pbar = tqdm(total=total_combinations, desc="Processing combinations")

    # Main evaluation loop
    processed_same_dataset = (
        set()
    )  # Track which ind_datasets we've processed for misclassification/selective

    for ind_dataset in datasets_ind:
        if ind_dataset == DatasetName.TINY_IMAGENET:
            datasets_ood = datasets_ood_tiny_imagenet_
        else:
            datasets_ood = datasets_ood_

        for ood_dataset in datasets_ood:
            process_uncertainty_additive_measure(
                ind_dataset,
                ood_dataset,
                metrics_list,
                prediction_data,
                results,
                pbar,
                processed_same_dataset,
            )

    pbar.close()

    # Convert results to DataFrame and save
    df = pd.DataFrame(results)

    # Create output filename with EntropicOT hyperparameters
    output_filename = "additive_baseline.csv"
    df.to_csv(output_filename, index=False)
    print(f"\nResults saved to {output_filename}")
    print(f"Total rows: {len(df)}")

    # Print summary statistics
    print("\n=== SUMMARY ===")
    print(f"Unique ind_datasets: {df['ind_dataset'].nunique()}")
    print(f"Unique ood_datasets: {df['ood_dataset'].nunique()}")
    print(f"Unique measures: {df['measure'].nunique()}")
    print(f"Problem types: {df['problem_type'].unique()}")

    return df

In [7]:
df = main()

Loading predictions for all datasets...


Processing combinations: 100%|██████████| 12/12 [00:00<00:00, 61.19it/s]


[Errno 2] No such file or directory: '../resources/results_cleaned/cifar10/risk_logscore_bayesrisk_outer_T_1.0/cifar10/cifar10_cifar10_risk_logscore_bayesrisk_outer_T_1.0.npz'
[Errno 2] No such file or directory: '../resources/results_cleaned/cifar100/risk_logscore_bayesrisk_outer_T_1.0/cifar100/cifar100_cifar100_risk_logscore_bayesrisk_outer_T_1.0.npz'
[Errno 2] No such file or directory: '../resources/results_cleaned/tiny_imagenet/risk_logscore_bayesrisk_outer_T_1.0/tiny_imagenet/tiny_imagenet_tiny_imagenet_risk_logscore_bayesrisk_outer_T_1.0.npz'

Results saved to additive_baseline.csv
Total rows: 60

=== SUMMARY ===
Unique ind_datasets: 3
Unique ood_datasets: 7
Unique measures: 1
Problem types: ['ood_detection' 'misclassification_detection' 'selective_prediction']


In [8]:
df

,ind_dataset,ood_dataset,measure,gname,ensemble_group,problem_type,roc_auc,average_precision,accuracy,aurc,acc_cov_auc,coverage_at_1pct_error,coverage_at_2pct_error,coverage_at_5pct_error,n_ind_samples,n_ood_samples,n_correct,n_incorrect,ensemble_accuracy
0,cifar10,cifar100,naive_additive,NaN,0,ood_detection,0.914882,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.963750
1,cifar10,cifar10,naive_additive,NaN,0,misclassification_detection,0.932489,0.299951,0.963750,NaN,NaN,NaN,NaN,NaN,7200,NaN,6939.0,261.0,0.963750
2,cifar10,cifar10,naive_additive,NaN,0,selective_prediction,NaN,NaN,0.963750,0.003383,0.996478,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.963750
3,cifar10,cifar100,naive_additive,NaN,1,ood_detection,0.913518,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.964306
4,cifar10,cifar10,naive_additive,NaN,1,misclassification_detection,0.929915,0.287672,0.964306,NaN,NaN,NaN,NaN,NaN,7200,NaN,6943.0,257.0,0.964306
5,cifar10,cifar10,naive_additive,NaN,1,selective_prediction,NaN,NaN,0.964306,0.003427,0.996434,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.964306
6,cifar10,cifar100,naive_additive,NaN,2,ood_detection,0.914880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.963333
7,cifar10,cifar10,naive_additive,NaN,2,misclassification_detection,0.934204,0.318327,0.963333,NaN,NaN,NaN,NaN,NaN,7200,NaN,6936.0,264.0,0.963333
8,cifar10,cifar10,naive_additive,NaN,2,selective_prediction,NaN,NaN,0.963333,0.003454,0.996407,0.000139,0.000139,0.000139,7200,NaN,NaN,NaN,0.963333
9,cifar10,cifar100,naive_additive,NaN,3,ood_detection,0.911795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7200,10000.0,NaN,NaN,0.964167
